# Extra-descriptors stratified performance — EU (snapshot 14y)

Where do the hand-crafted **extra descriptors** (Bakketun et al. 2026-style
land-cover / topography features) help, and how does that interact with the
TESSERA lat16 latents? Compares the four EU arms per variable:

| arm | folder stem |
|---|---|
| baseline (+elev+mTPI) | `*_snap_bilinear_baseline_mtpi_wd` |
| baseline + extra | `*_snap_bilinear_baseline_mtpi_extradesc_wd` |
| TESSERA lat16 | `*_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd` |
| lat16 + extra | `*_snap_vae_lat16_plus_extradesc_concat_with_elev_mtpi_no_static_wd` |

Per-station MAE comes from each run's `test_station_errors.npz` (seeds
averaged; no re-evaluation). Strata come from the **GEE descriptor CSV
itself** (`station_extra_descriptors.csv`) — the same land-cover fractions /
topography stats the extra arms consume, so strata match the paper's surface
types (town, forest, coast, mountain, …). Station sets are identical across
all four arms (verified below), so per-stratum comparisons are exact.

Caveat: the baseline pair trains **with** ERA5 static fields, the TESSERA
pair without (inherited from the shortlisted parents) — within-pair deltas
are the cleanest reads.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

from tessera_downscaling.paths import processed_dir, training_runs_dir

RUN_ROOT = training_runs_dir("snapshot_14y_eu")
DESC_CSV = processed_dir("station_extra_descriptors.csv")
SEEDS = [42, 123, 456]

# Arm -> (colour from experiments.yaml, folder stem per variable). Colours are
# the repo-wide entity colours; identity is never colour-alone below (hatching
# marks the +extra arms, and every figure carries a legend).
ARMS = {
    "baseline": dict(
        colour="#555555",
        hatch=None,
        t2m="t2m_snap_bilinear_baseline_mtpi_wd",
        wind="wind_truncnormal_snap_bilinear_baseline_mtpi_wd",
    ),
    "baseline+ex": dict(
        colour="#8c564b",
        hatch="//",
        t2m="t2m_snap_bilinear_baseline_mtpi_extradesc_wd",
        wind="wind_truncnormal_snap_bilinear_baseline_mtpi_extradesc_wd",
    ),
    "lat16": dict(
        colour="#9467bd",
        hatch=None,
        t2m="t2m_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd",
        wind="wind_truncnormal_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd",
    ),
    "lat16+ex": dict(
        colour="#e377c2",
        hatch="//",
        t2m="t2m_snap_vae_lat16_plus_extradesc_concat_with_elev_mtpi_no_static_wd",
        wind="wind_truncnormal_snap_vae_lat16_plus_extradesc_concat_with_elev_mtpi_no_static_wd",
    ),
}
ARM_ORDER = list(ARMS)
VARIABLES = ["t2m", "wind"]

In [ ]:
# Per-station MAE per (variable, arm), averaged over seeds. Only stations the
# test split actually evaluates (count > 0) are kept.
def load_arm(variable: str, folder: str) -> pd.DataFrame:
    frames = []
    for seed in SEEDS:
        npz = RUN_ROOT / f"{folder}_seed{seed}" / "test_station_errors.npz"
        d = np.load(npz, allow_pickle=True)
        keep = d[f"{variable}_station_count"] > 0
        frames.append(
            pd.DataFrame(
                {
                    "station_id": d["station_ids"][keep].astype(str),
                    "mae": d[f"{variable}_station_mae"][keep],
                }
            )
        )
    df = pd.concat(frames)
    agg = df.groupby("station_id", as_index=False).agg(
        mae=("mae", "mean"), n_seeds=("mae", "count")
    )
    assert (agg.n_seeds == len(SEEDS)).all(), "station missing in some seed"
    return agg.drop(columns="n_seeds")


per_station = {}  # (variable, arm) -> df[station_id, mae]
for variable in VARIABLES:
    sets = {}
    for arm, spec in ARMS.items():
        per_station[(variable, arm)] = load_arm(variable, spec[variable])
        sets[arm] = set(per_station[(variable, arm)].station_id)
    ref = sets["baseline"]
    assert all(s == ref for s in sets.values()), (
        f"{variable}: station sets differ between arms"
    )
    print(
        f"{variable}: {len(ref)} evaluated test stations, identical across all 4 arms"
    )

In [ ]:
# Strata from the GEE descriptor CSV — thresholds chosen to mirror the
# paper's surface types; tune here and re-run. Strata may overlap (a coastal
# town counts as both), matching the paper's Fig. 5 convention.
desc = pd.read_csv(DESC_CSV, dtype={"station_id": str}).set_index("station_id")

STRATA = {
    "all": desc.index == desc.index,
    "urban": desc.built_frac >= 0.30,
    "cropland": desc.crop_frac >= 0.50,
    "forest": desc.forest_frac >= 0.50,
    "open lowveg": desc.lowveg_frac >= 0.50,
    "coast/water": desc.water_frac >= 0.20,
    "mountain": desc.elev_std >= 200.0,  # 12.5 km kernel elevation std
    "flat lowland": desc.elev_std < 50.0,
}
strata_df = pd.DataFrame(
    {name: mask for name, mask in STRATA.items()}, index=desc.index
)
counts = {
    v: {
        n: int(strata_df.loc[list(per_station[(v, "baseline")].station_id), n].sum())
        for n in STRATA
    }
    for v in VARIABLES
}
print(pd.DataFrame(counts).rename_axis("stratum (evaluated stations)"))

In [ ]:
# Stratum x arm table: mean per-station MAE + the three deltas that matter.
def stratum_table(variable: str) -> pd.DataFrame:
    rows = []
    base = per_station[(variable, "baseline")].set_index("station_id")
    for name in STRATA:
        in_stratum = strata_df.index[strata_df[name]]
        row = {"stratum": name}
        for arm in ARM_ORDER:
            df = per_station[(variable, arm)].set_index("station_id")
            sel = df.loc[df.index.intersection(in_stratum), "mae"]
            row[arm] = sel.mean()
            row["n"] = len(sel)
        row["Δ base+ex %"] = (row["baseline+ex"] / row["baseline"] - 1) * 100
        row["Δ lat16+ex %"] = (row["lat16+ex"] / row["lat16"] - 1) * 100
        row["Δ tessera %"] = (row["lat16"] / row["baseline"] - 1) * 100
        rows.append(row)
    return pd.DataFrame(rows).set_index("stratum")


tables = {v: stratum_table(v) for v in VARIABLES}
for v in VARIABLES:
    print(f"===== {v} — mean per-station MAE by stratum =====")
    print(tables[v].round(3).to_string())
    print()

In [ ]:
# Grouped bars: MAE by stratum for the four arms (one panel per variable,
# separate y-axes — never dual-axis). Hatch marks the +extra arms so the
# pairing survives grayscale / CVD; 2px white edges keep bars separable.
fig, axes = plt.subplots(len(VARIABLES), 1, figsize=(9.5, 7.5), sharex=True)
strata_names = list(STRATA)
x = np.arange(len(strata_names))
width = 0.2
for ax, variable in zip(axes, VARIABLES, strict=False):
    t = tables[variable]
    for i, arm in enumerate(ARM_ORDER):
        spec = ARMS[arm]
        ax.bar(
            x + (i - 1.5) * width,
            t[arm],
            width * 0.92,
            color=spec["colour"],
            hatch=spec["hatch"],
            edgecolor="white",
            linewidth=1.2,
            label=arm,
        )
    ax.set_ylabel(f"{variable} MAE")
    ax.set_title(f"{variable}: per-station MAE by surface stratum (mean over seeds)")
    ax.grid(axis="y", alpha=0.25, linewidth=0.6)
    ax.set_axisbelow(True)
axes[0].legend(ncol=4, frameon=False, loc="upper left")
axes[-1].set_xticks(x, strata_names, rotation=20, ha="right")
fig.tight_layout()
plt.show()

In [ ]:
# The complementarity read: % MAE change from adding the extra descriptors,
# within each family, per stratum. Negative = descriptors help. Marker shape
# distinguishes families; a shared zero line gives polarity.
fig, axes = plt.subplots(1, len(VARIABLES), figsize=(10, 4.2), sharey=True)
for ax, variable in zip(axes, VARIABLES, strict=False):
    t = tables[variable]
    y = np.arange(len(t.index))
    ax.axvline(0, color="#999999", linewidth=1)
    ax.plot(
        t["Δ base+ex %"],
        y,
        "s",
        ms=8,
        color=ARMS["baseline+ex"]["colour"],
        label="baseline family (+extra vs baseline)",
    )
    ax.plot(
        t["Δ lat16+ex %"],
        y,
        "o",
        ms=8,
        color=ARMS["lat16+ex"]["colour"],
        label="TESSERA family (+extra vs lat16)",
    )
    ax.set_yticks(y, t.index)
    ax.set_xlabel("Δ MAE from adding extra descriptors (%)")
    ax.set_title(variable)
    ax.grid(axis="x", alpha=0.25, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.invert_yaxis()
axes[0].legend(frameon=False, loc="lower left", fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# Continuous view: MAE vs descriptor quantile bins (quintiles over evaluated
# stations), four arms as lines. Where the descriptor-vs-error relationship
# lives, without threshold choices.
BIN_DESCRIPTORS = ["built_frac", "forest_frac", "water_frac", "elev_std"]
fig, axes = plt.subplots(
    len(VARIABLES), len(BIN_DESCRIPTORS), figsize=(12.5, 6.2), sharex="col"
)
for r, variable in enumerate(VARIABLES):
    ids = per_station[(variable, "baseline")].station_id
    d = desc.loc[ids]
    for c, col in enumerate(BIN_DESCRIPTORS):
        ax = axes[r, c]
        # rank-based bins; duplicate edges collapse for zero-heavy fractions
        bins = pd.qcut(d[col].rank(method="first"), 5, labels=False)
        for arm in ARM_ORDER:
            spec = ARMS[arm]
            mae = per_station[(variable, arm)].set_index("station_id").loc[ids, "mae"]
            means = mae.groupby(bins.values).mean()
            ax.plot(
                means.index + 1,
                means.values,
                marker="o",
                ms=4,
                linewidth=2,
                color=spec["colour"],
                linestyle="--" if spec["hatch"] else "-",
                label=arm,
            )
        if r == 0:
            ax.set_title(col, fontsize=10)
        if c == 0:
            ax.set_ylabel(f"{variable} MAE")
        if r == len(VARIABLES) - 1:
            ax.set_xlabel("quintile (low → high)")
        ax.grid(alpha=0.25, linewidth=0.6)
        ax.set_axisbelow(True)
axes[0, 0].legend(frameon=False, fontsize=7, loc="best")
fig.tight_layout()
plt.show()

**Reading guide.** The bar panel gives absolute per-stratum error; the dot
panel is the headline — squares left of zero mean hand-crafted descriptors
help the no-TESSERA model in that stratum, circles left of zero mean they
still add on top of TESSERA (complementary information). The quintile grid
shows the same story continuously: where curves for the `+ex` arms (dashed)
separate from their solid parents, the descriptors carry signal; where
purple already sits below brown, TESSERA had it covered.

Strata are defined on the descriptors the extra arms *consume* — so a gap
closing in, say, the `urban` stratum is direct evidence the `built_frac`
input is doing the work (cf. Bakketun et al.'s 12% town improvement).